# Towards a McEliece-type scheme based on Preparata codes

In [15]:
def hamming_weight(v):
    """
    Return the Hamming weight of a vector.

    The Hamming weight is the number of non-zero entries of the vector.

    Args:
        v: A vector.

    Returns:
        int: The Hamming weight of v.
    """
    return sum(x != 0 for x in v)

def hamming_weight_for_polynomials(p):
    """
    Return the number of nonzero coefficients of a polynomial p.

    Args:
        p: A polynomial.
    """
    return sum(x != 0 for x in p.list())

def hamming_distance_for_polynomials(p, q):
    """
    Return the Hamming distance between two polynomials over the same ring,
    namely the Hamming weight of their sum. 

    Args:
        p: A polynomial.
        q: A polynomial over the same ring of p.
    """

    if p.parent() != q.parent():
        raise ValueError("The polynomials p and q are not defined over the same ring.")
    return hamming_weight_for_polynomials(p - q)

    
def vector_to_polynomial(v):
    """
    Return the representation of a vector of dimension n as a polynomial of 
    degree n-1 with coefficients in the same base field as the entries of the vector.

    The first entry of the vector is the coefficient of x^(n-1), while
    the last entry is the constant term.
    
    Args:
        v: A vector of entries (v_0, v_1, ... v_{n-1}) of dimension n.

    Returns:
        Polynomial: The polynomial v_0 x^(n-1) + v_1 x^(n-2) + ... + v_{n-1}.
        
    """
    K = v.base_ring()
    R.<x> = PolynomialRing(K)
    p = R(0)
    n = len(v)
    
    for j, u in enumerate(v):
        p += u * x**(n -1 - j)

    return p

def polynomial_to_vector(p, n):
    """
    Return the representation of a polynomial of degree at most n-1 as a vector
    of dimension n with entries in the same base field as the coefficients of
    the polynomial.

    The first entry of the vector is the coefficient of x^(n-1), while
    the last entry is the constant term.

    If the degree of the polynomial is strictly less than n-1, zeros are added 
    at the beginning so that the length of the output is still n.

    Args:
        p: A polynomial of degree at most n-1.
        n: An integer that is the desired length of the output vector.
    
    Returns:
        v: A vector of entries (v_0, v_1, ... v_{n-1}) of dimension n
        corresponding to v_0 x^(n-1) + ... + v_{n-1}.
    """
    if p.degree() >= n:
        raise ValueError("The degree of p is greater or equal than n")
        
    R = p.base_ring()

    coefficients = list(reversed(p.list()))
    coefficients = [0] * (n - len(coefficients)) + coefficients
   
    return vector(R, coefficients)

def preparata_to_vector(preparata, w_list):
    """
    Convert a triple w_list = [w0(x), w, w1(x)] into a binary vector
    of dimension n = 2^m -1.
    
    w0(x) and w1(x) are polynomials in Am_1, while w is a binary parameter.
    """
    if len(w_list) != 3:
        raise ValueError("w_list is not a triple")

    w0, w, w1 = w_list

    if w not in (0, 1):
            raise ValueError("w must be either 0 or 1")

    if w0.parent() != preparata.Am_1 or w1.parent() != preparata.Am_1:
            raise ValueError("w0 and w1 must belong to A_{m-1}")
        
    n = preparata.n_bch1
  
    w_vector = vector(preparata.F2,
                      list(polynomial_to_vector(w0.lift(), n))
                      +[w]
                      + list(polynomial_to_vector(w1.lift(), n)))
    
    if len(w_vector) != 2**preparata.m -1:
        raise ValueError("The vector w has not the right dimension")

    return w_vector

def vector_to_preparata(preparata, vector):
    """
    Given a vector of length 2**m -1, return a list [a(x), i, b(x)], where 
    a(x) and b(x) belong to Am_1 and i is a binary parameter.
    """
    if len(vector) != (2**preparata.m - 1):
        raise ValueError("the length of vector is not 2**m - 1")

    n = preparata.n_bch1

    a = preparata.Am_1(vector_to_polynomial(vector[:n]))

    i = preparata.F2(vector[n])

    b = preparata.Am_1(vector_to_polynomial(vector[n + 1:]))

    return [a, i, b]

## Binary Goppa codes and the usual McEliece cryptosystem

First we create the class irreducible binary Goppa codes. We fix $q=2$ and require $n=2^m$, $1 <t <n$ as inputs.

We want to obtain $\Gamma(L, g)$ binary irreducible Goppa code with support $L= \{\gamma_0, \gamma_1, \dots, \gamma_{n-1}\} = \mathbb{F}_{2^m}$ and Goppa polynomial $g(x) \in \mathbb{F}_{2^m}[x]$ irreducible of degree $t$. The polynomial $g(x)$ can be either given as an input or randomly selected (as in the original McEliece cryptosystem). 

The parameters of $\Gamma(L, g)$ are:
- $n= 2^m$;
- $k \ge n -mt$;
- $d_{min} \ge 2t +1$, so that the error-correction capability is at least $t$.

A generator matrix, a parity-check matrix and the list of codewords are attributes of the code.

In [16]:
class BinaryGoppaCode:
    """
    Create a binary irreducible Goppa code starting from a given set of parameters. 
    
    The support is the whole extension field of F_2 and g is an irreducible polynomial of
    degree t, either provided by the user or selected randomly.
    """

    def __init__(self, n, t, g=None):
        """
        Initialise a binary Goppa code and store its parameters as attributes.

        Check that the input parameters are valid, i.e. that n= 2^m and 1 < t < n. 
        If the polynomial g is provided, check that it has degree t and is monic and 
        irreducible over F_{2^m}.
        If g not provided, randomly select an irreducible polynomial of degree t over F_{2^m}.

        Compute a generator matrix G and parity-check matrix H.

        Args:
            n(int): The desired length of the code. It must be a power of 2.
            t(int): The error-correction capability of the code and the degree of the Goppa polynomial.
            g(univariate polynomial): The irreducible Goppa polynomial. If not provided,
            one is selected randomly.

        """

        
        # Check parameters
        self.m = ZZ(n).log(2)
        if 2**self.m != n:
            raise ValueError("n must be a power of 2")
        
        if not (1< t < n):
            raise ValueError("t must be greater than 1 and less than n")

        self.n = n
        self.t = t

        # Construct F_2, F_{2^m} and polynomial ring over F_{2^m}
        self.F2 = GF(2)
        self.F = GF(2**self.m, name = 'a')
        R.<x> = PolynomialRing(self.F)

        # check g if provided, else select g
        if g is not None:
            if g not in R:
                raise ValueError("g must be a polynomial over F_{2^m}")
            
            if g.degree() != self.t:
                raise ValueError("g must have degree t")

            if not g.is_monic():
                raise ValueError("g must be monic")
                
            if not g.is_irreducible():
                raise ValueError("g must be irreducible over F_{2^m}")
            
            self.g = g
        else:
            while True: # g has to be monic and irreducible of degree t
                g = x**self.t + R.random_element(degree=self.t - 1)
                if g.is_irreducible():
                    break
            self.g = g

        #construct the support L = F_{2^m}
        self.L = list(self.F)
        
        # Construct H with coefficients in F_{2^m}
        Hbar = matrix(self.F, self.t, self.n)

        
        for j, gamma in enumerate(self.L): #the cicle constructs the matrix by columns
            inv_g = 1 / self.g(gamma)
        
            for i in range(self.t):
                Hbar[i, j] = inv_g * gamma**i

        #construct H with coefficients in F_2
        H = matrix(self.F2, self.t*self.m, self.n)

        for i in range(self.t):
            for j in range(self.n):
                binary_representation = self.binary_representation(Hbar[i, j], self.m)
        
                for r in range(self.m):
                    H[i*self.m + r, j] = binary_representation[r]

        self.H = H
        self.Hbar = Hbar

        #compute dimension of the code
        self.k = self.n - self.H.rank()

        #construct the generator matrix G
        self.G = matrix(self.F2, self.H.right_kernel().basis())
        if self.G.nrows() != self.k or self.G.ncols() != self.n:
            raise ValueError("The generator matrix does not have \
            the expected dimensions.")
        

    def codewords(self):
        """
        Return a list containing all the codewords of self.
        """
        codewords = []
        Fk = list(GF(2**self.k, name = 'b'))
        for v in Fk:
            u = self.binary_representation(v, self.k)
            codewords.append(u * self.G)
        return codewords

    def binary_representation(self, z, s):
        """
        Return the binary vector representation of an element of F_{2^s}.

        Args:
            s(int): The integer identifying the extension F_{2^s}
            z(element in the extension): An element of F_{2^s}.

        Returns:
            List: A list containing all the coefficients of z seen as a polynomial
            in a, ordered increasingly from degree 0 to the degree
            of the polynomial representation of z, plus some zeros
            so that the list has length s.
        
        """
        coefficients = z.polynomial().list() 
        coefficients += [0] * (s - len(coefficients))
        return vector(self.F2, coefficients)

    def is_in_Goppa(self, v):
        """
        Given a binary vector of dimension n, return True if it is a codeword
        of the Goppa code, False otherwise.
        """
        if len(v) != self.n:
            raise ValueError("The vector v does not have dimension n")
            
        if v.base_ring() != self.F2:
            raise ValueError("The vector v is not binary")

        return self.H * v == 0
            
    def generate_codeword(self):
        """
        Return a codeword of the Goppa code generated randomly.
        """
        u = random_vector(self.F2, self.k)

        return u * self.G

We want to simulate a McEliece cryptosystem. \
We are given a binary irreducible Goppa code $\Gamma(L, g)$, with parameters $(n = 2^m, k)$ and error-correction capability $t$. We want to randomly select
- $S \in \mathbb{F}^{k \times k}$ non singular;
- $P \in \mathbb{F}^{n \times n}$ permutation matrix.
Then we want to construct $G' = SGP \in  \mathbb{F}^{k \times n}$ generator matrix of the public code, which we will use for encrytpion.

In [17]:
class McEliece:
    """
    Implement the McEliece public-key cryptosystem based on a binary irreducible
    Goppa code.
    
    Given a binary Goppa code, the class randomly generates a non-singular
    scrambling matrix S and a permutation matrix P, and constructs the public
    generator matrix G' = SGP. S and P can also be given as an input.
    
    It also provides the data required for encryption
    and decryption, including Patterson's decoding algorithm. 
    """
    def __init__(self, code, S=None, P=None):
        """
        Initialise a McEliece cryptosystem from a binary Goppa code.

        Args:
            code(BinaryGoppaCode): A binary Goppa code with parameters (n, k) and 
            error-correction capability t.
        """
        if not isinstance(code, BinaryGoppaCode):
            raise ValueError("The code is not an object of the class BinaryGoppaCode")

        self.n = code.n
        self.m = code.m
        self.k = code.k
        self.t = code.t
        self.G = code.G
        self.L = code.L
        
        self.F2 = GF(2)
        self.F = GF(2**self.m, name = 'a')
        R.<x> = PolynomialRing(self.F)
        self.g = code.g

        # Randomly generate S, a non-singular matrix, and P, a permutation matrix.
        if S is not None:
            if S.base_ring() != self.F2:
                raise ValueError("S must be a matrix over F_2")
            if not (S.ncols() == self.k and S.nrows() == self.k):
                raise ValueError("The matrix S has not dimensions k x k")
            if S.is_invertible():
                self.S = S 
            else: 
                raise ValueError("The matrix S is not invertible")
        else:
            self.S = self._generate_S()

        if P is not None:
            if P.base_ring() != self.F2:
                raise ValueError("P must be a matrix over F_2")
            if not (P.ncols() == self.n and P.nrows() == self.n):
                raise ValueError("The matrix P has not dimensions n x n")
            if not all(sum(P.row(i)) == 1 for i in range(self.n)):
                raise ValueError("P is not a permutation matrix")
            if not all(sum(P.column(j)) == 1 for j in range(self.n)):
                raise ValueError("P is not a permutation matrix")
            self.P = P
        else:
            self.P = self._generate_P()

        #Compute the public generator matrix G' of the code
        self.G_public = self.S * self.G * self.P

    def _generate_S(self):
        """
        Generate a random invertible matrix of size k over F_2.
        """
        while True:
            S = random_matrix(self.F2, self.k, self.k)
            if S.is_invertible():
                return S

    def _generate_P(self):
        """
        Generate a random permutation matrix of size n over F_2.
        """
        permutation = Permutations(self.n).random_element()
        P = matrix(self.F2, self.n, self.n, sparse=True)
    
        for i, j in enumerate(permutation):
            P[i, j - 1] = 1
        return P

    def print_private_key(self):
        """
        Print the private key of the McEliece cryptosystem.
    
        The private key consists of the generator matrix G of the Goppa code,
        the Goppa polynomial g, the non-singular scrambling matrix S, and the
        permutation matrix P.
        """
        print("Private key:")
        print(f"G =\n{self.G}")
        print(f"g = {self.g}")
        print(f"S =\n{self.S}")
        print(f"P =\n{self.P}")

    def print_public_key(self):
        """
        Print the public key of the McEliece cryptosystem.
    
        The public key consists of the public generator matrix G' and the
        parameters n, k, and t of the public code.
        """
        print("Public key:")
        print(f"G' =\n{self.G_public}")
        print(f"n = {self.n}")
        print(f"k = {self.k}")
        print(f"t = {self.t}")

    def private_key(self):
        """
        Return the private key of the McEliece cryptosystem.
    
        Returns:
            tuple: A tuple containing the generator matrix G, the Goppa
                polynomial g, the scrambling matrix S, and the permutation
                matrix P.
        """
        return self.G, self.g, self.S, self.P

    def public_key(self):
        """
        Return the public key of the McEliece cryptosystem.
    
        Returns:
            tuple: A tuple containing the public generator matrix G',
                and the parameters n, k, and t.
        """
        return self.G_public, self.n, self.k, self.t
        
    def random_error(self):
        """
        Return a random error vector of dimension n and Hamming weight at most t.
        """
        while True:
            e = random_vector(self.F2, self.n)
            if hamming_weight(e) <= self.t:
                return e
        
    def encrypt(self, M):
        """
        Given a binary vector M of dimension k, compute the ciphertext obtained
        by multiplying M by the public generator matrix and adding a random
        error vector e of Hamming weight at most t.
        """
        if M.base_ring() != self.F2 or len(M) != self.k:
            raise ValueError("M must be a binary vector of dimension k")
        e = self.random_error()
        return M * self.G_public + e

    def _reduce_mod_g(self, p):
        """
        Reduce the univariate polynomial p in F[x] modulo g.

        Returns:
             Polynomial: The remainder of p modulo g, of degree less than deg(g).
        """
        R.<x> = PolynomialRing(self.F)
        
        if p.parent() != R:
            raise ValueError("The polynomial p is not in the ring F[x]")

        #Returns the remainder of the division between p and g   
        return p % self.g 
            

    def _patterson_decode(self, c):
        """
        Given a binary vector c of dimension n, obtained as c = codeword + error,
        recover the error vector of Hamming weight at most t.

        Solves the key equation S(x)sigma(x) = sigma'(x) mod g(x).

        The roots of the univariate polynomial sigma(x) identify the non-zero
        entries of the error vector. sigma(x) has degree at most t.
        """
        
        if c.base_ring() != self.F2 or len(c) != self.n:
            raise ValueError("c must be a binary vector of dimension n")
            
        #Compute the syndrome polynomial S(x)
        R = self.g.parent()
        x = R.gen()
        syndrome = 0
        
        for i, gamma in enumerate(self.L):
            if c[i] !=0:
                syndrome +=  inverse_mod(x + gamma, self.g)

        #If the syndrome polynomial is 0, no error has occurred and we return
        #the zero vector of dimension n
        if syndrome == 0:
            return vector(self.F2, self.n)

        h = self.g.xgcd(syndrome)[2]
        #h = inverse_mod(syndrome, self.g)

        #We want to find sigma such that syndrome * sigma = sigma' mod g(x)
        # sigma = alpha^2 + xbeta^2 
        
        if h == x :
            sigma = x
        else:
            d_squared = h + x
            
            K = R.quotient(self.g)
            z = R.gen()
            d = K(d_squared) ** (2**(self.m * self.t - 1))
            d = d.lift()

            #Need to solve the congruence d * beta = alpha mod g(x)
            #Extended Euclidean algorithm between g and d to find alpha and beta
            #alpha has degree <= t/2, beta has degree <= (t-1)/2

            #Initialise step 0 and 1 of the extended euclidean algorithm and check
            #if the conditions on the degrees are already satisfied

            r_0, r_1 = self.g, d
            s_0, s_1 = R(1), R(0)
            t_0, t_1 = R(0), R(1)
            
            if (r_1.degree() <= self.t // 2 and 
                t_1.degree() <= (self.t - 1) // 2):
                alpha = r_1
                beta = t_1

            while r_1 != 0: 
                q = r_0.quo_rem(r_1)[0]
                r_2 = r_0 - q * r_1
                s_2 = s_0 - q * s_1
                t_2 = t_0 - q * t_1 
            
                if (r_2.degree() <= self.t // 2 and 
                t_2.degree() <= (self.t - 1) // 2):
                    alpha = r_2
                    beta = t_2
                    break

                r_0, r_1 = r_1, r_2
                s_0, s_1 = s_1, s_2
                t_0, t_1 = t_1, t_2

        sigma = alpha ** 2 + x * beta ** 2
        error_positions = [0] * self.n

        for j, gamma in enumerate(self.L):
            if sigma(gamma) == 0:
                error_positions[j] = 1
                
        return vector(self.F2, error_positions)       

    def decrypt(self, ciphertext):
        """
        Given a binary vector ciphertext of dimension n, recover the message M,
        a binary vector of dimension k, such that C = MG' + e, where e
        is an error vector of dimension n and Hamming weight at most t.
        """
        #Check the input is correct 
        if ciphertext.base_ring() != self.F2 or len(ciphertext) != self.n:
            raise ValueError("The ciphertext must be a binary vector of dimension n")
            
        #The inverse of P permutation matrix is the transpose
        c = ciphertext * self.P.transpose() 
        error = self._patterson_decode(c)
        codeword = c - error
        
        #Solve the system uG = codeword, where u = mS
        u = self.G.solve_left(codeword)
        return u * self.S.inverse()
        

## Preparata codes and the McEliece-type scheme

We define the class of Preparata codes, which are binary and nonlinear. For every $m \ge 4$ even, there exist a Preparata code with:
- length $n = 2^m - 1$;
- dimension $k = 2^m -2m$;
- error-correction capability $t=2$.
  
We use as ring 
$$ A_{m-1} = \mathbb{F}_2[x] / (x^{2^{m - 1}-1} + 1)$$
where the polynomials have degree at most $2^{m-1}-2$ and at most $2^{m-1}-1$ nonzero coefficients.

Codewords are then formed by adjointing to a certain linear code, obtained by juxtaposition and superposition of BCH codes of length $2^{m-1}-1$, a subset of its cosets, hence the nonlinear structure. 

We denote by $\mathcal{K}_m$ the Preparata code, by $\mathcal{C}_m$ the underlying linear code and consider the following set
$$
\mathcal{U} =\{[q(x), 0, q(x)f(x)] \mid q(x)= ax^j, a=0,1 \text{ and } j=0, 1, \dots, 2^{m - 1}-2 \}
$$
and $f(x) = f(x)^2$ in $A_{m-1}$.

With this notation,
$$\mathcal{K}_m = \mathcal{C}_m + \mathcal{U}.$$

In [18]:
class CmCode:
    """
    Create a binary linear code C_m, starting from an even integer m greater or 
    equal than 4. The minimum distance of the code is at least 6.
    """
    def __init__(self, m):
        """
        Create the code C_m starting from an even integer m greater than 
        or equal to 4.
        """
        if m < 4 or (m % 2) != 0:
            raise ValueError("The integer m must be an even number greater than or \
            equal to four.")

        self.m = m
        self.F2 = GF(2)
        
        #Construction of the single error-correcting BCH code {m(x)} of length
        # 2^{m-1}-1 and generator polynomial g1 of degree m-1. Only the generator
        #polynomial is needed, i.e. the minimal polynomial of a
        self.n_bch1 = 2**(self.m - 1) - 1
        self.F1 = GF(self.n_bch1 + 1)
        self.a = self.F1.gen()
        
        R.<x> = PolynomialRing(self.F2)
        self.Am_1 = R.quotient(x ** (self.n_bch1) + 1)
        
        self.g1 = self.a.minimal_polynomial()

        #Construction of the code {s(x)} whose generator polynomial has as roots
        #a, a^3, 1 and whose minimum distance is  6, such that {s(x)} < {m(x)}
        #and g1 | g3.
        
        self.g3 = lcm([(self.a).minimal_polynomial(),
                      (self.a**3).minimal_polynomial(), 
                      x + 1])
        

        self.u = ( x**self.n_bch1 + 1 ) // (x + 1)
        

    def is_codeword(self, v):
        """
        Check whether the triple v = [a(x), i, b(x)] is a codeword of C_m.
    
        Each codeword is represented by a triple [a(x), i, b(x)], where:
        - a(x) = m(x) is a codeword of the BCH single error-correcting code;
        - i is a binary parameter;
        - b(x) = m(x) +(m(1) + i)u(x) + s(x), with s(x) a codeword of the cyclic
            code having a, a^3 and 1 as roots of the generator polynomial g3.

        Args:
             v (list): A triple [a(x), i, b(x)], where a(x) and b(x)
                belong to A_{m-1} and i is in {0, 1}.
        """
        if len(v) != 3:
            raise ValueError("v must be a triple")
            
        a, i, b = v
        
        if i not in (0, 1):
            raise ValueError("i must be either 0 or 1")

        if a.parent() != self.Am_1 or b.parent() != self.Am_1:
            raise ValueError("a and b must belong to A_{m-1}")

        a_polynomial = a.lift()
        b_polynomial = b.lift()
        s = b_polynomial - a_polynomial + (a_polynomial(1) + i) * self.u
        
        #Check that the polynomial a is in the code generated by g1 and s in the one
        #generated by g3
        return (a_polynomial % self.g1 == 0) and (s % self.g3 == 0)

    def generate_v(self):
        """
        Generate a random codeword v of C_m in the form
        [m(x), i, m(x) + (m(1) + i)u(x) + s(x)], where m(x) is a codeword of
        the code generated by g1, s(x) is a codeword of the code generated by
        g3, and i is in {0, 1}.
    
        Returns:
            list: A list representing the triple
            [m(x), i, m(x) + (m(1) + i)u(x) + s(x)].
        """
        i = self.F2.random_element()
        R = self.g1.parent()

        #Generate r_a with deg(r_a) < n_bch1 - deg(g1)
        #so that the a = r_a * g1 is a codeword of the code generated by g1.
        r_a = R.random_element(degree=self.n_bch1 - self.g1.degree() - 1)
        a = r_a * self.g1

        r_s = R.random_element(degree=self.n_bch1 - self.g3.degree() - 1)
        s = r_s * self.g3
        
        b = a + (a(1) + i)* self.u + s

        return [self.Am_1(a), i, self.Am_1(b)]

        
        

In [19]:
C_m_code = CmCode(6)

In [20]:
assert C_m_code.g1(C_m_code.a) == 0
assert C_m_code.g1.degree() == C_m_code.m - 1
assert C_m_code.g3 % C_m_code.g1 == 0

In [21]:
zero = C_m_code.Am_1(0)
assert C_m_code.is_codeword([zero, 0, zero]) == True

a = C_m_code.g1
s = C_m_code.g3
i = 0

b = a + (a(1) + i) * C_m_code.u + s

assert C_m_code.is_codeword([C_m_code.Am_1(a), i, C_m_code.Am_1(b)]) == True

In [22]:
class PreparataCode:
    """
    Create a binary Preparata code for an even integer m greater than
    or equal to 4. 
    
    The code is nonlinear, is defined over F_2, corrects up to two errors and has
    parameters (2^m -1, 2^m - 2*m) for m >= 4 even.
    """
    #prima devo creare il codice lineare C_m a partire dai BCH
    #poi finalmente si costruisce preparata
    def __init__(self, m):
        """
        Create the Preparata code for an even integer m greater than
        or equal to 4.
        """
        if m < 4 or (m % 2) != 0:
            raise ValueError("The integer m must be an even number greater than or \
            equal to four.")

        self.m = m
        self.F2 = GF(2)
        
        #Construct the linear code C_m
        self.Cm_code = CmCode(self.m)

        self.n_bch1 = self.Cm_code.n_bch1
        self.F1 = self.Cm_code.F1
        self.a = self.Cm_code.a
        self.Am_1 = self.Cm_code.Am_1
        
        
        self.g1 = self.Cm_code.g1
        self.g3 = self.Cm_code.g3

        #Define the matrices H1, H3, U encoding the condition a, a^3, 1 is a root of
        #g3 respectively. Together they constitute the parity-check matrix of
        #the cyclic code generated by g3. They are matrices over F_2^{m-1}

        self.H1 = matrix(self.F1,
            [[self.a**j for j in range(self.n_bch1 - 1, -1, -1)]])
        
        self.H3 = matrix(self.F1,
            [[(self.a**3)**j for j in range(self.n_bch1 - 1, -1, -1)]])
        
        self.U = matrix(self.F1,
            [[1 for _ in range(self.n_bch1)]])
        
        self.H = self.H1.stack(self.H3).stack(self.U)

        
        self.u =  self.Cm_code.u

        R.<x> = PolynomialRing(self.F2)
        
        #Construct phi(x) and the corresponding idempotent f(x)
        self.phi = (x**(self.n_bch1) + 1) // self.g1
        self.f = self.find_idempotent()

    def find_idempotent(self):
        """
        Find and return a polynomial f(x) = x^s phi(x) such that
        f(x)^2 = f(x) in Am_1.
        """
        f = self.Am_1(self.phi)
        x = self.Am_1.gen()
        
        for _ in range(self.n_bch1):
            if f**2 == f:
                return f
            f *= x

        raise ValueError("No idempotent polynomial was found.")

    def is_in_U(self, u):
        """
        Check whether the vector u = [a(x), i, b(x)] is of the form
        [q(x), 0, q(x)f(x)], where q(x) = ax^j with a=0,1 and j=0, .. 2^{m-1}-2.
        The polynomials q(x) are minimum weight coset leaders for the code
        {m(x)} generated by g1.

        Args:
             u (list): A triple [a(x), i, b(x)], where a(x) and b(x)
                belong to A_{m-1} and i is in {0, 1}.
            
        """
        if len(u) != 3:
            raise ValueError("u must be a triple")
        a, i, b = u
        
        if i != 0:
            return False
            
        if a == 0:
            return b == 0

        if hamming_weight_for_polynomials(a) != 1:
            return False
        
        return b == a * self.f  

    def generate_u(self, j=None):
        """
        Generate a vector u =[q(x), 0, q(x)f(x)] where q(x) = ax^j with a in {0,1}
        and j=0, .. 2^{m-1}-2.

        The polynomials q(x) are minimum weight coset leaders for the code
        {m(x)} generated by g1.

        Args:
            j(int, optional): An integer such that 0 <= j <= 2^{m-1}-2. If None,
                j is chosen randomly.

        Returns:
            list: A list representing the triple [q(x), 0, q(x)f(x)].
        """
        x = self.Am_1.gen()
        
        if j is not None:
            if j < 0 or j > self.n_bch1 - 1:
                raise ValueError("The integer j is not in the right interval")  
            a = x ** j
        else: 
            j = ZZ.random_element(self.n_bch1)
            a = ZZ.random_element(2) * x ** j
            
        b = a * self.f

        return [a, self.F2(0), b]

    def generate_w(self):
        """
        Generate a random codeword w of the Preparata code K_m in the form
        w = v + u, where v is a codeword of C_m and u is a vector in U.
    
        Returns:
            list: A list representing the triple
            [a(x) + q(x), i, b(x) + q(x)f(x)].
        """
        v = self.Cm_code.generate_v()
        u = self.generate_u()

        return [v[0] + u[0], v[1], v[2] + u[2]]

    def is_in_Km(self, w):
        """
        Check whether the triple w = a(x), i, b(x)] is a codeword of K_m.
    
        A vector w belongs to K_m if it can be written as w = v + u,
        where v is a codeword of C_m and u belongs to U.
    
        Args:
           w (list): A triple [a(x), i, b(x)], where a(x) and b(x)
                belong to A_{m-1} and i is in {0, 1}.
    
        Returns:
            bool: True if [a(x), i, b(x)] is a codeword of K_m,
            and False otherwise.
        """
        if len(w) != 3:
            raise ValueError("w must be a triple")
    
        a, i, b = w
    
        if i not in (0, 1):
            raise ValueError("i must be either 0 or 1")
    
        if a.parent() != self.Am_1 or b.parent() != self.Am_1:
            raise ValueError("a and b must belong to A_{m-1}")
    
        x = self.Am_1.gen()
    
        #if q = 0, u = [0,0,0] and we check if w = v = [a, i, b] is a codeword of Cm
        if self.Cm_code.is_codeword(w):
            return True
    
        for j in range(self.n_bch1):
            q = x**j
            u = [q, 0, q * self.f]
            v = [a + u[0], i, b + u[2]]
    
            if self.Cm_code.is_codeword(v):
                return True
    
        return False
        
    def characteristics(self, h):
        """
        Compute and return the characteristics of the polynomial h(x), namely the
        triple [beta1, beta3, c] obtained by multiplying its coefficients
        vector by the parity-check matrix H of of the code generated by g3(x).
        Characteristics(h) = (h(a), h(a^3), h(1)).
        """
        if h.parent() != self.Am_1:
            raise ValueError("The polynomial h is not in the ring Am_1")
            
        h_vector = polynomial_to_vector(h.lift(), self.n_bch1)

        #The product matrix * h_vector is a tuple with one element,
        #so we take the element using [0]; in this way the output is an
        #element of F1
        beta1 = (self.H1 * h_vector)[0]
        beta3 = (self.H3 * h_vector)[0]
        c = (self.U * h_vector)[0]
        
        return [beta1, beta3, c]

    def syndrome(self, r_list):
        """
        Compute the syndrome of the received vector r_list = [r0(x), r, r1(x)],
        that is the sum w + e of a codeword w and an error e.

        The syndrome is the quadruple [sigma0, sigma1, sigma, d] where:
        - sigma0 = H1 * r0^T
        - sigma1 = H1 * r1^T
        - sigma = H3 * (r0 + r1)^T
        - d = r + U * r1^T.

        Args:
            r_list (list): A triple [r0(x), r, r1(x)], where r0(x) and r1(x)
                belong to A_{m-1} and r is in {0, 1}.

        Returns:
            list: The syndrome [sigma0, sigma1, sigma, d].
        """
        if len(r_list) != 3:
            raise ValueError("r_list must be a triple")
    
        r0, r, r1 = r_list
    
        if r not in (0, 1):
            raise ValueError("r must be either 0 or 1")
    
        if r0.parent() != self.Am_1 or r1.parent() != self.Am_1:
            raise ValueError("r0 and r1 must belong to A_{m-1}")

        r0_vector = polynomial_to_vector(r0.lift(), self.n_bch1)
        r1_vector = polynomial_to_vector(r1.lift(), self.n_bch1)
        r_sum_vector = r0_vector + r1_vector

        sigma0 = (self.H1 * r0_vector)[0]
        sigma1 = (self.H1 * r1_vector)[0]

        sigma = (self.H3 * r_sum_vector)[0]
        d = r + (self.U * r1_vector)[0]

        return [sigma0, sigma1, sigma, d]
        
    def correct_error(self, r_list):
        """
        Decode the received vector r_list and compute the corresponding error
        vector according to the decoding rules of the Preparata code.
    
        Args:
            r_list (list): A triple [r0(x), r, r1(x)], where r0(x) and r1(x)
                belong to A_{m-1} and r belongs to {0, 1}.
    
        Returns:
            list: The corrected codeword and the corresponding error vector.
        """
        sigma0, sigma1, sigma, d = self.syndrome(r_list)

        # quantities used to distinguish the cases
        s = sigma0 + sigma1 
        rho = sigma + s ** 3
        a = rho + sigma0**3
        b = rho + sigma1**3

        x = self.Am_1.gen()
        c = [self.Am_1(0), 0, self.Am_1(0)] #c= [c0(x), c, c1(x)]

        if (a == 0) and (b == 0):
            if d == 0: #r_list is already a codeword
                return r_list, c
            if d == 1: # the error has weight 1 and is in the second component
                c[1] = 1

        #Error of weight 1 in c0
        elif (a != 0) and (b == 0): #Rule 1
            h = s.log(self.a)
            c[0] = x**h

            #c = d + c1(1) but in this case c1(1) = 0 (1) = 0
            c[1] = d 

        #Error of weight 1 in c1
        elif (a == 0) and (b != 0): #Rule 1
            h = s.log(self.a)
            c[2] = x**h 

            #c = d + c1(1) but in this case c1(1) = x^h (1) = 1 
            c[1] = d + 1

        
        elif (a != 0) and (b != 0):
            if (d == 1): #Rule 2, one error in each of c0 and c1
                c[1] = 0
                
                k = (sigma + sigma0 * sigma1 * s).nth_root(3)
                k0 = (sigma1 + k).log(self.a)
                k1 = (sigma0 + k).log(self.a)
                
                c[0] = x ** k0
                c[2] = x ** k1
            
            elif (d == 0): #Rule 3, two errors in either c0 or c1
                if (s != 0): 
                    c[1] = 0

                    R1.<z> = PolynomialRing(self.F1)

                    #Only one between p0 and p1 has roots in F1
                    p0 = z**2 + s*z + a/s #j=0
                    p1 = z**2 + s*z + b/s #j=1

                    roots0 = p0.roots()  #p0.roots.() return a list of tuples

                    if len(roots0) == 2:
                        c[0] = 0
                        
                        z1 = roots0[0][0]
                        z2 = roots0[1][0]
                        
                        k1 = z1.log(self.a)
                        k2 = z2.log(self.a)
                        c[2] = x**k1 + x**k2

                    else:            
                        c[2] = 0
                        roots1 = p1.roots()
                        assert len(roots1) == 2
                        
                        z1 = roots1[0][0]
                        z2 = roots1[1][0]

                        k1 = z1.log(self.a)
                        k2 = z2.log(self.a)
                        c[0] = x**k1 + x**k2
                        

                else: #Rule 4
                     raise ValueError( "The received vector is at distance \
                     at least 3 from every codeword.")

        corrected_word = [r_list[0] + c[0], r_list[1] + c[1], r_list[2] + c[2]]
        return corrected_word, c


    def generate_error(self):
        """
        Generate a random correctable error, represented as [e0(x), e, e1(x)]. 

        Possible configurations of weight 1 are:
        - [x^s, 0, 0];
        - [0, 0, x^s];
        - [0, 1, 0].

        Possible configurations of weight 2 are:
        - [x^s + x^t, 0, 0], with s different from t;
        - [0, 0, x^s + x^t], with s different from t;
        - [x^s, 0, x^t];
        - [x^s, 1, 0];
        - [0, 1, x^s].
        
        The zero-error is not considered.
        """
        x = self.Am_1.gen()
        error_type = ZZ.random_element(8)

        if error_type == 0:
            #Error of weight 1 in the first polynomial component
            j = ZZ.random_element(self.n_bch1)
            return [x**j, self.F2(0), self.Am_1(0)]

        elif error_type == 1:
            #Error of weight 1 in the third polynomial component
            j = ZZ.random_element(self.n_bch1)
            return [self.Am_1(0), self.F2(0), x**j]

        elif error_type == 2:
            # Error in the scalar component
            return [self.Am_1(0), self.F2(1), self.Am_1(0)]

        elif error_type == 3:
            # Two errors in the first polynomial component
            j = ZZ.random_element(self.n_bch1)
            k = ZZ.random_element(self.n_bch1)
            while k == j:
                k = ZZ.random_element(self.n_bch1)
            return [x**j + x**k, self.F2(0), self.Am_1(0)]

        elif error_type == 4:
            # Two errors in the third polynomial component
            j = ZZ.random_element(self.n_bch1)
            k = ZZ.random_element(self.n_bch1)
            while k == j:
                k = ZZ.random_element(self.n_bch1)
            return [self.Am_1(0), self.F2(0), x**j + x**k]

        elif error_type == 5:
            # One error in each polynomial component
            j = ZZ.random_element(self.n_bch1)
            k = ZZ.random_element(self.n_bch1)
            return [x**j, self.F2(0), x**k]

        elif error_type == 6:
            #One error in the first polynomial component and one in the
            #scalar component
            j = ZZ.random_element(self.n_bch1)
            return [x**j, self.F2(1), self.Am_1(0)]       

        else:
            #One error in the scalar component and one in the third entry
            j = ZZ.random_element(self.n_bch1)
            return [self.Am_1(0), self.F2(1), x**j] 
            
        

In [23]:
class McEliecePreparata:
    """
    Implement a McEliece-type cryptosystem based on a binary Preparata code.
    
    Since binary Preparata codes are nonlinear, they do not admit a
    generator matrix as in the classical McEliece construction. The
    plaintext is therefore a codeword of the Preparata code. The scheme
    randomly generates a permutation matrix P, which can also be provided
    as input, and encrypts a codeword w by computing
    
        c = wP + e,
    
    where e is a randomly generated error vector of Hamming weight at most 2.
    
    The class also provides the procedures required for decryption,
    including the Preparata decoding algorithm.
    """
    def __init__(self, code, P=None):
        """
        Initialise a McEliece-type cryptosystem from a binary Preparata code.

        Args:
            code(PreparataCode): A binary Preparata code with parameter m and 
            error-correction capability t = 2. m must be >= 4 and even.
            P (matrix, optional): A permutation matrix of size N x N over F_2.
                If not provided, a random permutation matrix is generated.
        """
        if not isinstance(code, PreparataCode):
            raise ValueError("The code is not an object of the class PreparataCode")

        self.code = code
        self.m = code.m
        self.t = 2
        self.N = 2**self.m -1
        
        self.F2 = GF(2)

        R.<x> = PolynomialRing(self.F2)
        self.Am_1 = code.Am_1

        self.u = code.u
        self.phi = code.phi
        self.f = code.f
        
        #randomly generate the permutation matrix P or use the input
        if P is not None:
            if P.base_ring() != self.F2:
                raise ValueError("P must be a matrix over F_2")
            if not (P.ncols() == self.N and P.nrows() == self.N):
                raise ValueError("The matrix P has not dimensions N x N")
            if not all(sum(P.row(i)) == 1 for i in range(self.N)):
                raise ValueError("P is not a permutation matrix")
            if not all(sum(P.column(j)) == 1 for j in range(self.N)):
                raise ValueError("P is not a permutation matrix")
            self.P = P
        else:
            self.P = self._generate_P()

    def _generate_P(self):
        """
        Generate a random permutation matrix of size N over F_2.
        """
        permutation = Permutations(self.N).random_element()
        P = matrix(self.F2, self.N, self.N, sparse=True)
    
        for i, j in enumerate(permutation):
            P[i, j - 1] = 1
        return P

    def print_private_key(self):
        """
        Print the private key of the McEliece-type cryptosystem.
        """
        print("Private key:")
        print(f"P =\n{self.P}")

    def print_public_parameters(self):
        """
        Print the public parameters of the McEliece-type cryptosystem.
    
        The public parameters are N, m, and t of the code.
        """
        print("Public key:")
        print(f"N = {self.N}")
        print(f"m = {self.m}")
        print(f"t = {self.t}")

    def private_key(self):
        """
        Return the private key of the McEliece-type cryptosystem.
        """
        return self.P

    def public_parameters(self):
        """
        Return the public parameters of the McEliece-type encryption scheme.
    
        Returns:
            tuple: A tuple containing the parameters N, m, and t.
        """
        return self.N, self.m, self.t
        
    def encrypt(self, w):
        """
        Given a Preparata codeword w = [w0(x), i , w1(x)], compute the ciphertext
        obtained by multiplying w by the permutation matrix P and adding a random
        correctable error vector e of Hamming weight at most 2.

        Args:
            w (list): A list containing two polynomials in Am_1 and a binary integer,
                that is a codeword of the Preparata code.

        Returns:
            list: A list representing the ciphertext.
        """
        if not self.code.is_in_Km(w):
            raise ValueError("The vector w is not a Preparata codeword")

        w_vector = preparata_to_vector(self.code, w)
            
        e_list = self.code.generate_error()
        e = preparata_to_vector(self.code, e_list)
        
        return vector_to_preparata(self.code, w_vector * self.P + e)         

    def decrypt(self, ciphertext):
        """
        Given a ciphertext represented as list, recover the message w,
        represented as a Preparata codeword, such that C = w*P + e, where e
        is an error vector of dimension N and Hamming weight at most 2.
        """
        cipher_vector = preparata_to_vector(self.code, ciphertext)
        #Check the input is correct 
        if cipher_vector.base_ring() != self.F2 or len(cipher_vector) != self.N:
            raise ValueError("The ciphertext must be a binary vector of dimension N")
            
        #The inverse of P permutation matrix is the transpose
        c_prime = cipher_vector * self.P.transpose() 
        c = vector_to_preparata(self.code, c_prime)
        codeword, error = self.code.correct_error(c)

        return codeword
        
        

In [24]:
preparata = PreparataCode(6)

w = preparata.generate_w()
u = preparata.generate_u()
v = preparata.Cm_code.generate_v()

In [25]:
assert preparata.is_in_Km(w)
assert preparata.is_in_U(u)
assert preparata.Cm_code.is_codeword(v)

In [26]:
for _ in range(100):
    v = preparata.Cm_code.generate_v()
    u = preparata.generate_u()
    w = [v[0] + u[0], v[1], v[2] + u[2]]

    assert preparata.Cm_code.is_codeword(v)
    assert preparata.is_in_U(u)
    assert preparata.is_in_Km(w)

In [27]:
for _ in range(100):
    w = preparata.generate_w()
    assert preparata.is_in_Km(w)

In [28]:
# Dimensions
assert preparata.H1.nrows() == 1
assert preparata.H1.ncols() == preparata.n_bch1

assert preparata.H3.nrows() == 1
assert preparata.H3.ncols() == preparata.n_bch1

assert preparata.U.nrows() == 1
assert preparata.U.ncols() == preparata.n_bch1

assert preparata.H.nrows() == 3
assert preparata.H.ncols() == preparata.n_bch1


# Entries belong to F1
assert all(entry in preparata.F1
           for entry in preparata.H1.list())

assert all(entry in preparata.F1
           for entry in preparata.H3.list())

assert all(entry in preparata.F1
           for entry in preparata.U.list())

assert all(entry in preparata.F1
           for entry in preparata.H.list())

# H1
for j in range(preparata.n_bch1):
    assert preparata.H1[0, j] == preparata.a**(preparata.n_bch1 - 1 - j)

# H3
for j in range(preparata.n_bch1):
    assert preparata.H3[0, j] == (preparata.a**3)**(preparata.n_bch1 - 1 - j)
    
# U
assert all(entry == 1 for entry in preparata.U.list())

# H
assert preparata.H == (preparata.H1.stack(preparata.H3).stack(preparata.U))

assert preparata.H[0] == preparata.H1[0]
assert preparata.H[1] == preparata.H3[0]
assert preparata.H[2] == preparata.U[0]

In [29]:
R = preparata.g3.parent()

for _ in range(100):
    r = R.random_element(degree=preparata.n_bch1 - preparata.g3.degree() - 1)
    s = r * preparata.g3
    s_vector = polynomial_to_vector(s, preparata.n_bch1)
    
    assert preparata.H * s_vector == 0

In [30]:
for _ in range(100):
    h = R.random_element(degree=preparata.n_bch1 - 1)
    h = preparata.Am_1(h)

    beta1, beta3, c = preparata.characteristics(h)

    assert beta1 == h.lift()(preparata.a)
    assert beta3 == h.lift()(preparata.a**3)
    assert c == h.lift()(1)

In [31]:
for _ in range(100):
    r = R.random_element(
        degree=preparata.n_bch1 - preparata.g3.degree() - 1
    )
    s = r * preparata.g3
    s = preparata.Am_1(s)

    assert preparata.characteristics(s) == [0, 0, 0]

In [32]:
r1 = preparata.generate_w()
r2 = preparata.Cm_code.generate_v()

assert preparata.is_in_Km(r1)
assert preparata.is_in_Km(r2)

w1, c1 = preparata.correct_error(r1)
w2, c2 = preparata.correct_error(r2)

assert r1 == w1
assert r2 == w2

assert c1 == [preparata.Am_1(0), 0, preparata.Am_1(0)]
assert c2 == [preparata.Am_1(0), 0, preparata.Am_1(0)]


In [33]:
#Test rule 1: one error in the first entry
w3 = preparata.generate_w()

x = preparata.Am_1.gen()
e3 = [x**5, 0, preparata.Am_1(0)]

r3 = [w3[0] + e3[0], w3[1], w3[2] + e3[2]]

decoded3, error3 = preparata.correct_error(r3)

assert decoded3 == w3
assert error3 == e3

In [34]:
#Test rule 1: one error in the third entry
w4 = preparata.generate_w()
e4 = [preparata.Am_1(0), 0, x**5]

r4 = [w4[0] + e4[0], w4[1] + e4[1], w4[2] + e4[2]]

decoded4, error4 = preparata.correct_error(r4)

assert decoded4 == w4
assert error4 == e4

In [35]:
#Test rule 2: one error in both the first and third entries
w5 = preparata.generate_w()
e5 = [x**5, 0, x**11]

r5 = [w5[0] + e5[0], w5[1] + e5[1], w5[2] + e5[2]]

decoded5, error5 = preparata.correct_error(r5)

assert decoded5 == w5
assert error5 == e5

In [36]:
#Test rule 3: 2 errors in the first entry
w6 = preparata.generate_w()
e6 = [x**5 + x**11, 0, preparata.Am_1(0)]

r6 = [w6[0] + e6[0], w6[1], w6[2]]

decoded6, error6 = preparata.correct_error(r6)

assert decoded6 == w6
assert error6 == e6

In [37]:
#Test rule 3: 2 errors in the third entry
w7 = preparata.generate_w()
e7 = [preparata.Am_1(0), 0, x**5 + x**11]

r7 = [w7[0] + e7[0], w7[1], w7[2] + e7[2]]

decoded7, error7 = preparata.correct_error(r7)

assert decoded7 == w7
assert error7 == e7

In [38]:
#The error has weight 1 and is in the second component
w8 = preparata.generate_w()
e8 = [preparata.Am_1(0), preparata.F2(1), preparata.Am_1(0)]

r8 = [w8[0], w8[1] + preparata.F2(1), w8[2]]

decoded8, error8 = preparata.correct_error(r8)

assert decoded8 == w8
assert error8 == e8

In [39]:
#test with an error of weight 1 in the first and second entries
w9 = preparata.generate_w()
e9 = [x**5, preparata.F2(1), preparata.Am_1(0)]

r9 = [w9[0] + e9[0], w9[1] + e9[1], w9[2]]

decoded9, error9 = preparata.correct_error(r9)

assert decoded9 == w9
assert error9 == e9

In [40]:
#test with an error of weight 1 in the third and second entries
w10 = preparata.generate_w()
e10 = [preparata.Am_1(0), preparata.F2(1), x**5]

r10 = [w10[0], w10[1] + e10[1], w10[2] + e10[2]]

decoded10, error10 = preparata.correct_error(r10)

assert decoded10 == w10
assert error10 == e10

In [41]:
while True:
    r0 = preparata.Am_1(
        R.random_element(degree=preparata.n_bch1 - 1)
    )
    r1 = preparata.Am_1(
        R.random_element(degree=preparata.n_bch1 - 1)
    )
    r = preparata.F2.random_element()

    r = [r0, r, r1]

    sigma0, sigma1, sigma, d = preparata.syndrome(r)

    s = sigma0 + sigma1
    rho = sigma + s**3
    a = rho + sigma0**3
    b = rho + sigma1**3

    if (a != 0) and (b != 0) and (d == 0) and (s == 0):
        break

try:
    preparata.correct_error(r)
    assert False, "Rule 4 should have been triggered."
except ValueError as error:
    print(error)

The received vector is at distance                      at least 3 from every codeword.


In [42]:
#test the decoder with casual errors of weight <=2
for _ in range(1000):
    w = preparata.generate_w()
    e = preparata.generate_error()

    r = [w[0] + e[0], w[1] + e[1], w[2] + e[2]]

    decoded, error = preparata.correct_error(r)

    assert decoded == w
    assert error == e

In [43]:
N = 2**preparata.m - 1

#define the permutation matrix
permutation = list(range(N))
shuffle(permutation)
P = matrix(GF(2), N, N, lambda i, j: 1 if permutation[i] == j else 0)
P_inv = P.inverse()

true_count = 0
false_count = 0

for _ in range(1000):
    w = preparata.generate_w()
    
    w_vector = preparata_to_vector(preparata, w)
    w_permuted = w_vector * P
    w_permuted = vector_to_preparata(preparata, w_permuted)
    
    if preparata.is_in_Km(w_permuted):
        true_count += 1
    else:
        false_count += 1

print("True:", true_count)
print("False:", false_count)


True: 1
False: 999


In [44]:
w17 = preparata.generate_w()
e17 = preparata.generate_error()

r17 = [w17[0] + e17[0], w17[1] + e17[1], w17[2] + e17[2]]

w17_vector = preparata_to_vector(preparata, w17)
e17_vector = preparata_to_vector(preparata, e17)

w17_encoded = w17_vector * P + e17_vector 

w17_decoded = vector_to_preparata(preparata, w17_encoded * P_inv)

decoded, error = preparata.correct_error(w17_decoded)

decoded == w17

True

In [45]:
true_count = 0
false_count = 0

for _ in range(1000):
    # Generate a codeword and a correctable error
    w = preparata.generate_w()
    e = preparata.generate_error()

    # Convert them to binary vectors
    w_vector = preparata_to_vector(preparata, w)
    e_vector = preparata_to_vector(preparata, e)

    # Encoding
    encoded = w_vector * P + e_vector

    # Undo the permutation and convert back to Preparata representation
    received = vector_to_preparata(preparata, encoded * P_inv)

    # Decode
    decoded, error = preparata.correct_error(received)

    # Check whether the original codeword was recovered
    if decoded == w:
        true_count += 1
    else:
        false_count += 1

print("True:", true_count)
print("False:", false_count)

True: 1000
False: 0


In [46]:
#Test the decoding procedure without the knowledge of P
results = []

num_experiments = 10
num_trials = 1000
total_trials = num_experiments * num_trials

for _ in range(num_experiments):
    correct_count = 0 #the decoder recovers w
    wrong_count = 0 #the decoder recovers another Preparata codeword
    failure_count = 0 #the decoder is not able to recover a codeword

    for _ in range(num_trials):
        # Generate a codeword and a correctable error
        w = preparata.generate_w()
        e = preparata.generate_error()
    
        # Convert them to binary vectors
        w_vector = preparata_to_vector(preparata, w)
        e_vector = preparata_to_vector(preparata, e)
    
        # Encoding
        encoded = w_vector * P + e_vector
    
        # Use preparata representation for encoded vector
        received = vector_to_preparata(preparata, encoded)
    
        # Try to decode without knowing P
        try:
            decoded, error = preparata.correct_error(received)

            if decoded == w:
                correct_count += 1
            else:
                wrong_count += 1
    
        except ValueError:
            failure_count += 1

    results.append([correct_count, wrong_count, failure_count])

total_correct = sum(r[0] for r in results)
total_wrong = sum(r[1] for r in results)
total_failure = sum(r[2] for r in results)

correct_average = total_correct / num_experiments
wrong_average = total_wrong / num_experiments
failure_average = total_failure / num_experiments

correct_percentage = 100.0 * total_correct / total_trials
wrong_percentage = 100.0 * total_wrong / total_trials
failure_percentage = 100.0 * total_failure / total_trials

print(f"Average correct: {float(correct_average):.2f}")
print(f"Average wrong: {float(wrong_average):.2f}")
print(f"Average failure: {float(failure_average):.2f}")

print(f"Correct: {correct_percentage:.2f}%")
print(f"Wrong: {wrong_percentage:.2f}%")
print(f"Failure: {failure_percentage:.2f}%")

Average correct: 0.00
Average wrong: 986.80
Average failure: 13.20
Correct: 0.00%
Wrong: 98.68%
Failure: 1.32%


In [47]:
## A comparison between Goppa and Preparata construction of McEliece schemes

We now try to compare the McEliece cryptosystem with a binary Goppa code with the McEliece-type system with Preparata codes. 

The parameters for the Goppa code will be:
- $n= 1024 = 2^{10} $;
- $t = 2$;
- $k \ge n - m \cdot t = 1024 - 20 = 1004 $;
which is the same length proposed by McEliece, but with an error-correction capability that is very small compared to the $t= 50$ of the original paper. The choice is due to the fact we want a fair comparision between the two codes and similar parameters.

The parameters for the Preparata code will be:
- $n = 2 ^ m - 1 = 2^{10}-1 = 1023 $, so $m = 10$;
- $t = 2$;
- $k = 2^m - 2m = 1024 - 20 = 1004$

In [48]:
#Initialise both codes and cryptosystems
m = 10
n = 2**m
t = 2

Goppa = BinaryGoppaCode(n, t)
Preparata = PreparataCode(m)

Mceliece = McEliece(Goppa)
Mceliece_type = McEliecePreparata(Preparata)

In [49]:
# Test the decoding procedure with Preparata code without the knowledge of P

results = []

num_experiments = 10
num_trials = 200
total_trials = num_experiments * num_trials

for _ in range(num_experiments):
    correct_count = 0   # the decoder recovers w
    wrong_count = 0     # the decoder recovers another Preparata codeword
    failure_count = 0   # the decoder fails to recover a codeword

    for _ in range(num_trials):
        # Generate a random Preparata codeword
        w = Preparata.generate_w()

        # Encrypt w using the secret permutation P
        ciphertext = Mceliece_type.encrypt(w)

        # Try to decode without knowing P
        try:
            decoded, error = Preparata.correct_error(ciphertext)

            if decoded == w:
                correct_count += 1
            else:
                wrong_count += 1

        except ValueError:
            failure_count += 1

    results.append([correct_count, wrong_count, failure_count])


# Compute averages
total_correct = sum(r[0] for r in results)
total_wrong = sum(r[1] for r in results)
total_failure = sum(r[2] for r in results)

correct_average = float(total_correct / num_experiments)
wrong_average = float(total_wrong / num_experiments)
failure_average = float(total_failure / num_experiments)

# Compute percentages
correct_percentage = float(100 * total_correct / total_trials)
wrong_percentage = float(100 * total_wrong / total_trials)
failure_percentage = float(100 * total_failure / total_trials)

print(f"Average correct: {correct_average:.2f}")
print(f"Average wrong: {wrong_average:.2f}")
print(f"Average failure: {failure_average:.2f}")

print(f"Correct: {correct_percentage:.2f}%")
print(f"Wrong: {wrong_percentage:.2f}%")
print(f"Failure: {failure_percentage:.2f}%")

Average correct: 0.00
Average wrong: 200.00
Average failure: 0.00
Correct: 0.00%
Wrong: 100.00%
Failure: 0.00%
